In [14]:
%%html
<div id="map" style="width: 100%; height: 500px; background: #f0f0f0; border-radius: 8px;"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
    // 1. Initialisation de la carte
    var map = L.map('map').setView([46.603354, 1.888334], 6);

    // 2. Ajout du fond de carte
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{z}.png', {
        maxZoom: 19,
        attribution: '&copy; CARTO'
    }).addTo(map);

    // 3. Chargement des données des régions françaises
    fetch('https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson')
        .then(response => response.json())
        .then(data => {
            var geojsonLayer = L.geoJson(data, {
                style: {
                    fillColor: '#3186cc',
                    weight: 1.5,
                    opacity: 1,
                    color: 'white',
                    fillOpacity: 0.4
                },
                onEachFeature: function(feature, layer) {
                    var nomRegion = feature.properties.nom;
                    layer.bindTooltip("<b>" + nomRegion + "</b>", { sticky: true });
                    
                    layer.on({
                        mouseover: function(e) {
                            e.target.setStyle({
                                fillColor: '#ff7800',
                                fillOpacity: 0.7
                            });
                        },
                        mouseout: function(e) {
                            geojsonLayer.resetStyle(e.target);
                        }
                    });
                }
            }).addTo(map);
        })
        .catch(err => console.error("Erreur de chargement :", err));

    // 4. Forcer le redimensionnement pour éviter l'affichage vide dans Jupyter
    setTimeout(function() {
        map.invalidateSize();
    }, 300);
</script>

In [15]:
pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
!pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import os
import folium
from folium.plugins import MarkerCluster
import pandas as pd

nom_fichier = 'Z:/SAE15/sup_support.txt'

if not os.path.exists(nom_fichier):
  print(f"Erreur : Le fichier est introuvable au chemin : {nom_fichier}")
else:
  print('Chargement du fichier en cours...')
  df = pd.read_csv(nom_fichier, sep=';', low_memory=False)


  def dms_to_dec(deg, mn, sc, ns_ew):
    try:
      dec = float(deg) + float(mn) / 60.0 + float(sc) / 3600.0
      if ns_ew in ['S', 'W']:
        dec = -dec
      return dec
    except:
      return None


  df['LAT'] = df.apply(
      lambda row: dms_to_dec(
          row['COR_NB_DG_LAT'],
          row['COR_NB_MN_LAT'],
          row['COR_NB_SC_LAT'],
          row['COR_CD_NS_LAT'],
      ),
      axis=1,
  )
  df['LON'] = df.apply(
      lambda row: dms_to_dec(
          row['COR_NB_DG_LON'],
          row['COR_NB_MN_LON'],
          row['COR_NB_SC_LON'],
          row['COR_CD_EW_LON'],
      ),
      axis=1,
  )

  df = df.dropna(subset=['LAT', 'LON'])

  # MODIFICATION ICI : Utilisation d'OpenStreetMap pour éviter le message "API KEY REQUIRED"
  m = folium.Map(location=[46.603354, 1.888334], zoom_level=6, tiles='OpenStreetMap')

  villes = [
      ('Paris', 48.8566, 2.3522),
      ('Lyon', 45.7640, 4.8357),
      ('Marseille', 43.2965, 5.3698),
      ('Toulouse', 43.6047, 1.4442),
      ('Bordeaux', 44.8378, -0.5792),
      ('Lille', 50.6292, 3.0573),
      ('Nantes', 47.2184, -1.5536),
      ('Strasbourg', 48.5734, 7.7521),
  ]
  for city, lat, lon in villes:
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size: 11px; font-weight: bold; color: #333; background: rgba(255,255,255,0.8); padding: 2px 5px; border-radius: 3px; border: 1px solid #ccc;">{city}</div>'
        ),
    ).add_to(m)

  print('Ajout des antennes sur la carte...')
  marker_cluster = MarkerCluster().add_to(m)

  for idx, row in df.head(5000).iterrows():
    tpo = str(row.get('TPO_ID', ''))
    color = 'blue'
    op_name = 'Bouygues Telecom'

    if '74' in tpo:
      color = 'orange'
      op_name = 'Orange'
    elif '32' in tpo:
      color = 'red'
      op_name = 'SFR'
    elif '29' in tpo:
      color = 'brown'
      op_name = 'Free Mobile'
    elif 'Multiple' in tpo or len(str(tpo)) > 5:
      color = 'black'
      op_name = 'Multi-opérateurs'

    popup_text = f"""
        <b>Support ID:</b> {row['SUP_ID']}<br>
        <b>Commune (INSEE):</b> {row['COM_CD_INSEE']}<br>
        <b>Opérateur:</b> {op_name}
        """

    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300),
    ).add_to(marker_cluster)

  legend_html = """
    <div style="
        position: fixed; 
        bottom: 50px; left: 50px; width: 220px; height: 150px; 
        background-color: white; z-index:9999; font-size:13px;
        border:2px solid grey; border-radius: 5px; padding: 10px;
        box-shadow: 0 0 15px rgba(0,0,0,0.2);">
        <b>Légende - Opérateurs</b><br>
        <i style="background:blue; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Bouygues<br>
        <i style="background:orange; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Orange<br>
        <i style="background:red; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> SFR<br>
        <i style="background:brown; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Free<br>
        <i style="background:black; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Multi-opérateurs<br>
    </div>
    """
  m.get_root().html.add_child(folium.Element(legend_html))

  output_file = 'Z:/SAE15/carte_antennes_france.html'
  m.save(output_file)
  print(f'Carte générée et enregistrée dans : {output_file}')

  display(m)

Chargement du fichier en cours...
Ajout des antennes sur la carte...
Carte générée et enregistrée dans : Z:/SAE15/carte_antennes_france.html


In [11]:
import os
import folium
from folium.plugins import MarkerCluster
import pandas as pd


def charger_donnees(chemin_fichier):
  """Charge le fichier CSV/TXT et convertit les coordonnées DMS en décimales."""
  print('Chargement et nettoyage des données...')
  df = pd.read_csv(chemin_fichier, sep=';', low_memory=False)

  def dms_to_dec(deg, mn, sc, ns_ew):
    try:
      dec = float(deg) + float(mn) / 60.0 + float(sc) / 3600.0
      if ns_ew in ['S', 'W']:
        dec = -dec
      return dec
    except:
      return None

  df['LAT'] = df.apply(
      lambda r: dms_to_dec(
          r['COR_NB_DG_LAT'],
          r['COR_NB_MN_LAT'],
          r['COR_NB_SC_LAT'],
          r['COR_CD_NS_LAT'],
      ),
      axis=1,
  )
  df['LON'] = df.apply(
      lambda r: dms_to_dec(
          r['COR_NB_DG_LON'],
          r['COR_NB_MN_LON'],
          r['COR_NB_SC_LON'],
          r['COR_CD_EW_LON'],
      ),
      axis=1,
  )

  return df.dropna(subset=['LAT', 'LON'])


def obtenir_couleur_operateur(row):
  """Identifie l'opérateur et associe une couleur de manière élargie

  (vérifie à la fois le TPO_ID et d'autres colonnes d'opérateurs si elles
  existent).
  """
  # On récupère toutes les informations textuelles possibles de la ligne
  tpo = str(row.get('TPO_ID', '')).lower()
  oper_nom = str(row.get('OPERATEUR', '')).lower()  # Si la colonne existe
  texte_ligne = str(row.values).lower()

  # 1. Multi-opérateurs (partagé)
  if (
      'multiple' in tpo
      or 'partag' in tpo
      or tpo.count(',') > 0
      or tpo.count(';') > 0
  ):
    return 'black', 'Multi-opérateurs'

  # 2. Orange (souvent code 20801 ou contient 'orange')
  if 'orange' in tpo or 'orange' in oper_nom or '20801' in tpo or '74' in tpo:
    return 'orange', 'Orange'

  # 3. SFR (souvent code 20810 ou contient 'sfr')
  if 'sfr' in tpo or 'sfr' in oper_nom or '20810' in tpo or '32' in tpo:
    return 'red', 'SFR'

  # 4. Free Mobile (souvent code 20815 ou contient 'free')
  if 'free' in tpo or 'free' in oper_nom or '20815' in tpo or '29' in tpo:
    return 'brown', 'Free Mobile'

  # 5. Bouygues Telecom (par défaut ou si identifié)
  if 'bouygues' in tpo or 'bouygues' in oper_nom or '20820' in tpo:
    return 'blue', 'Bouygues Telecom'

  # Si aucun test précis ne matche, on analyse par défaut les blocs de codes courants ANFR
  # (Vous pouvez ajuster ces critères selon ce que contient votre colonne TPO_ID)
  return 'blue', 'Bouygues Telecom'


def creer_carte(df):
  """Initialise la carte OpenStreetMap et ajoute les repères et antennes."""
  carte = folium.Map(
      location=[46.603354, 1.888334], zoom_level=6, tiles='OpenStreetMap'
  )

  # Repères des grandes villes
  villes = [
      ('Paris', 48.8566, 2.3522),
      ('Lyon', 45.7640, 4.8357),
      ('Marseille', 43.2965, 5.3698),
      ('Toulouse', 43.6047, 1.4442),
      ('Bordeaux', 44.8378, -0.5792),
      ('Lille', 50.6292, 3.0573),
      ('Nantes', 47.2184, -1.5536),
      ('Strasbourg', 48.5734, 7.7521),
  ]
  for nom, lat, lon in villes:
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size: 11px; font-weight: bold; color: #333; background: rgba(255,255,255,0.8); padding: 2px 5px; border-radius: 3px; border: 1px solid #ccc;">{nom}</div>'
        ),
    ).add_to(carte)

  print(
      'Ajout des antennes sur la carte (échantillon de 10 000 pour voir plus'
      ' de diversité)...'
  )
  cluster = MarkerCluster().add_to(carte)

  # On passe à 10000 lignes pour avoir un meilleur mélange d'opérateurs sur la France
  for _, row in df.head(10000).iterrows():
    color, op_name = obtenir_couleur_operateur(row)

    popup_text = f"""
        <b>Support ID:</b> {row['SUP_ID']}<br>
        <b>Commune (INSEE):</b> {row['COM_CD_INSEE']}<br>
        <b>Opérateur:</b> {op_name}
        """

    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300),
    ).add_to(cluster)

  return carte


# --- EXÉCUTION PRINCIPALE ---
chemin = 'Z:/SAE15/sup_support.txt'

if os.path.exists(chemin):
  df_antennes = charger_donnees(chemin)
  ma_carte = creer_carte(df_antennes)

  ma_carte.save('Z:/SAE15/carte_antennes_france.html')
  display(ma_carte)
else:
  print(f'Erreur : Fichier introuvable à l’emplacement {chemin}')

Chargement et nettoyage des données...
Ajout des antennes sur la carte (échantillon de 10 000 pour voir plus de diversité)...
